# dbAppsUA05: SQL Агрегації, GROUP BY та Excel

**Курс:** Застосування баз даних

У цьому зошиті ви навчитесь:
- Використовувати агрегатні функції (COUNT, SUM, AVG, MIN, MAX)
- Групувати дані з GROUP BY
- Фільтрувати групи з HAVING
- Розуміти різницю між WHERE та HAVING
- Експортувати результати в Excel

## Налаштування

In [ ]:
# Імпортуємо потрібні бібліотеки
import pandas as pd
import sqlite3

# Підключаємося до бази даних NBA
conn = sqlite3.connect('nba_5seasons.db')

print('Підключення до бази даних встановлено!')

## Розділ 1: Агрегатні функції (Aggregate Functions)

Агрегатні функції обчислюють один результат з набору значень:

- **COUNT(*)** — рахує всі рядки
- **SUM(column)** — сума значень
- **AVG(column)** — середнє значення
- **MIN(column)** / **MAX(column)** — мінімум / максимум
- **ROUND(value, decimals)** — округлення

In [ ]:
# COUNT(*) — рахуємо загальну кількість ігор
query = """
SELECT COUNT(*) AS totalGames
FROM team_game_stats;
"""

result = pd.read_sql(query, conn)
print(f"Загальна кількість ігор: {result['totalGames'][0]}")

In [ ]:
# SUM — загальна кількість очків за всі ігри
query = """
SELECT SUM(pts) AS totalPoints
FROM team_game_stats;
"""

result = pd.read_sql(query, conn)
print(f"Загальна кількість очків: {result['totalPoints'][0]:,.0f}")

In [ ]:
# AVG і ROUND — середня кількість очків за гру
query = """
SELECT ROUND(AVG(pts), 1) AS avgPointsPerGame
FROM team_game_stats;
"""

result = pd.read_sql(query, conn)
print(f"Середня кількість очків за гру: {result['avgPointsPerGame'][0]}")

In [ ]:
# MIN і MAX — знаходимо крайні значення
query = """
SELECT 
  MIN(pts) AS minPoints,
  MAX(pts) AS maxPoints
FROM player_season_stats;
"""

result = pd.read_sql(query, conn)
print(f"Мінімум очків за сезон: {result['minPoints'][0]}")
print(f"Максимум очків за сезон: {result['maxPoints'][0]}")

**Спробуй:** Обчисли середній відсоток влучань з гри (`fg_pct`) з таблиці `player_season_stats`, округлений до 2 знаків.

In [ ]:
# Спробуй: Середній fg_pct, округлений до 2 знаків

## Розділ 2: GROUP BY та псевдоніми (Aliases)

GROUP BY групує рядки за одним або кількома стовпцями. Зазвичай використовується з агрегатними функціями.

**AS** — створює псевдонім (alias) для перейменування стовпців у результаті.

In [ ]:
# GROUP BY — загальна кількість очків кожної команди
query = """
SELECT 
  team_id,
  SUM(pts) AS totalPoints
FROM team_game_stats
GROUP BY team_id
ORDER BY totalPoints DESC
LIMIT 5;
"""

result = pd.read_sql(query, conn)
print(result)
print("\nТоп 5 команд за загальною кількістю очків")

In [ ]:
# GROUP BY з кількома агрегатами та псевдонімами
query = """
SELECT 
  team_id AS teamID,
  COUNT(*) AS gamesPlayed,
  ROUND(AVG(pts), 1) AS avgPointsPerGame,
  MAX(pts) AS bestGame
FROM team_game_stats
GROUP BY team_id
ORDER BY avgPointsPerGame DESC
LIMIT 5;
"""

result = pd.read_sql(query, conn)
print(result)
print("\nТоп 5 команд за середніми очками за гру")

In [ ]:
# GROUP BY кілька стовпців — сезон і команда
query = """
SELECT 
  season,
  team_id,
  COUNT(*) AS gamesPlayed,
  ROUND(AVG(pts), 1) AS avgPoints
FROM team_game_stats
GROUP BY season, team_id
ORDER BY season DESC, avgPoints DESC
LIMIT 10;
"""

result = pd.read_sql(query, conn)
print(result)

**Спробуй:** Згрупуй `player_season_stats` за `player_id` і покажи кількість сезонів та середні очки за сезон (округлені до 1 знаку).

In [ ]:
# Спробуй: GROUP BY player_id

## Розділ 3: HAVING проти WHERE

- **WHERE** — фільтрує рядки **ДО** групування (окремі рядки)
- **HAVING** — фільтрує **ПІСЛЯ** групування (згруповані результати)

**Порядок:** SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY

In [ ]:
# WHERE — фільтруємо ДО групування
# Шукаємо середні очки тільки у виграних іграх
query = """
SELECT 
  team_id AS teamID,
  COUNT(*) AS wins,
  ROUND(AVG(pts), 1) AS avgPointsInWins
FROM team_game_stats
WHERE wl = 'W'
GROUP BY team_id
ORDER BY avgPointsInWins DESC
LIMIT 5;
"""

result = pd.read_sql(query, conn)
print(result)
print("\nТоп 5 команд за середніми очками у перемогах")

In [ ]:
# HAVING — фільтруємо ПІСЛЯ групування
# Команди з більш ніж 150 перемогами
query = """
SELECT 
  team_id AS teamID,
  COUNT(*) AS totalWins
FROM team_game_stats
WHERE wl = 'W'
GROUP BY team_id
HAVING COUNT(*) > 150
ORDER BY totalWins DESC;
"""

result = pd.read_sql(query, conn)
print(result)
print("\nКоманди з більш ніж 150 перемогами за 5 сезонів")

In [ ]:
# WHERE + GROUP BY + HAVING разом
query = """
SELECT 
  team_id AS teamID,
  COUNT(*) AS highScoringGames,
  ROUND(AVG(pts), 1) AS avgPoints
FROM team_game_stats
WHERE pts > 100
GROUP BY team_id
HAVING AVG(pts) > 110
ORDER BY avgPoints DESC;
"""

result = pd.read_sql(query, conn)
print(result)

**Спробуй:** Знайди гравців (з `player_season_stats`) які набирали в середньому більше 20 очків за сезон. Покажи player_id, кількість сезонів та середні очки.

In [ ]:
# Спробуй: Гравці з середніми очками > 20

## Розділ 4: Експорт у Excel

Можна експортувати результати SQL запитів у файли Excel (.xlsx).

In [ ]:
# Запитуємо статистику команд
query = """
SELECT 
  team_id AS teamID,
  COUNT(*) AS gamesPlayed,
  SUM(pts) AS totalPoints,
  ROUND(AVG(pts), 1) AS avgPointsPerGame
FROM team_game_stats
GROUP BY team_id
ORDER BY avgPointsPerGame DESC;
"""

# Зчитуємо дані у DataFrame
teamStats = pd.read_sql(query, conn)

# Експортуємо у Excel
teamStats.to_excel('team_statistics.xlsx', index=False, sheet_name='Team Stats')

print('Файл експортовано: team_statistics.xlsx')
print(f'Рядків експортовано: {len(teamStats)}')
teamStats.head()

**Спробуй:** Знайди топ-10 гравців за середніми очками за сезон і експортуй результат у файл `top_scorers.xlsx`.

In [ ]:
# Спробуй: Топ-10 гравців → Excel

## Підсумок

Ви навчились:

1. **Агрегатні функції:** COUNT, SUM, AVG, MIN, MAX, ROUND
2. **GROUP BY:** Групування за одним або кількома стовпцями з псевдонімами
3. **HAVING vs WHERE:** HAVING фільтрує групи, WHERE фільтрує рядки
4. **Експорт у Excel:** Базовий експорт з `.to_excel()`